In [1]:
import getpass
import selenium
import time
import os
import shutil
from pathlib import Path
from datetime import datetime, timedelta
import calendar
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.edge.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Alignment, Font
import subprocess
import sys


In [ ]:
def aplicar_formatacao_excel(caminho_arquivo, nome_aba):
    """
    Abre um arquivo Excel existente, aplica cores de fundo no cabeçalho,
    ajusta larguras das colunas e centraliza os textos.
    É fundamental usar wb.close() no final para liberar o arquivo no Windows.
    """
    wb = load_workbook(caminho_arquivo)
    ws = wb[nome_aba]
    
    # Define a cor de fundo Cinza para o cabeçalho
    fundo_cabeçalho = PatternFill(start_color="BFBFBF", fill_type="solid")
    alinhamento = Alignment(horizontal="center", vertical="center", wrap_text=True)

    # Aumenta a altura da primeira linha (Cabeçalho)
    ws.row_dimensions[1].height = 25
    
    # Aplica o estilo para cada coluna no cabeçalho
    for col in range(1, ws.max_column + 1):
        letra = get_column_letter(col)
        celula = ws.cell(row=1, column=col)
        celula.fill = fundo_cabeçalho
        celula.alignment = alinhamento
        
        # A coluna 'S' recebe uma largura maior, o resto recebe tamanho padrão 25
        ws.column_dimensions[letra].width = 75.6 if letra == 'S' else 20
        
    # Centraliza todas as células das linhas normais (linha 2 em diante)
    for linha in ws.iter_rows(min_row=2):
        for celula in linha: 
            celula.alignment = Alignment(vertical="center", wrap_text=True)
            
    # Salva e FECHA o arquivo para evitar erros de permissão ao mover
    wb.save(caminho_arquivo)
    wb.close() 

# ==========================================
# FUNÇÃO AUXILIAR DA JUSTIFICATIVA
# ==========================================
def padronizar_justificativa(df):
    if 'Status Revenue' not in df.columns:
        return df

    if 'Justificativa' not in df.columns:
        df['Justificativa'] = ''
    
    novos_valores = []
    for just, status in zip(df['Justificativa'], df['Status Revenue']):
        # Limpeza segura de valores nulos ou vazios
        just_str = '' if pd.isna(just) else str(just).strip().upper()
        status_str = '' if pd.isna(status) else str(status).strip().upper()
        
        eh_vazio = just_str in ('', 'NAN', 'NONE', 'NULL')
        
        if eh_vazio and status_str == 'APROVADO':
            novos_valores.append('Aprovada')
        elif eh_vazio and status_str == 'REPROVADO': # ou 'RECUSADO' dependendo do site
            novos_valores.append('Concorrência')
        else:
            novos_valores.append(str(just) if not pd.isna(just) else '')
            
    df['Justificativa'] = novos_valores
    return df


# ==========================================
# MOTOR DE CONCILIAÇÃO DOS ARQUIVOS EBUS
# ==========================================
import os
import shutil
import pandas as pd
from openpyxl import load_workbook

def processar_arquivos_relatorios(arquivo_original, destino, nome_mes, ano_atual, callback_progresso=None):
    """
    Função coração da regra de negócio EBUS.
    1. Lê a planilha web crua.
    2. Valida se veio vazia.
    3. Concatena colunas de Origem + Destino e aplica Justificativa.
    4. Processa a "Base Nova" (Histórico de Novo, Excluido, Presente).
    5. Processa a "Base Normal" (Mercado vazio, regra de volume).
    """
    
    if callback_progresso: callback_progresso(0.6, "Lendo dados originais...")
    df_novo = pd.read_excel(arquivo_original, engine='xlrd')

    df_novo = df_novo.dropna(how='all')

    # Verifica se o relatório veio vazio direto da mensagem do site
    if 'Não foi possivel obter dados com os parâmetros informados.' in df_novo.columns:
        if not df_novo.empty and df_novo['Não foi possivel obter dados com os parâmetros informados.'].iloc[0] is True:
            os.remove(arquivo_original)
            return
        elif df_novo.empty:
            os.remove(arquivo_original)
            return

    # =========================================================
    # TRATAMENTOS UNIVERSAIS (Aplicados a todas as bases)
    # =========================================================
    if 'Origem' in df_novo.columns and 'Destino' in df_novo.columns:
        df_novo['Concatenar Origem e Destino'] = df_novo['Origem'].astype(str) + " - " + df_novo['Destino'].astype(str)

    # Aplica as regras da Justificativa antes de começar as comparações
    df_novo = padronizar_justificativa(df_novo)

    # =========================================================
    # ESTRUTURA DE PASTAS E ARQUIVOS
    # =========================================================
    nome_oficial = f"Relatorio Revenue Completo - {nome_mes.capitalize()} {ano_atual}.xlsx"
    
    pasta_base = destino / "BASE"
    pasta_backup = destino / "BACKUP"
    pasta_base_nova = destino / "BASE NOVA"
    pasta_backup_base_nova = destino / "BACKUP NOVO"
    
    # Criando todas as pastas de uma vez de forma limpa
    for pasta in [pasta_base, pasta_backup, pasta_base_nova, pasta_backup_base_nova]:
        pasta.mkdir(parents=True, exist_ok=True)

    caminho_base = pasta_base / nome_oficial
    caminho_backup = pasta_backup / nome_oficial
    caminho_base_nova = pasta_base_nova / nome_oficial
    caminho_backup_base_nova = pasta_backup_base_nova / nome_oficial

    # =========================================================
    # PARTE 1: BASE NOVA (Mapeamento de Novo / Excluido / Presente)
    # =========================================================
    if callback_progresso: callback_progresso(0.7, "Processando Base Nova...")
    houve_alteracao_estado = True

    if caminho_base_nova.exists():
        df_base_estado_antiga = pd.read_excel(caminho_base_nova)
        
        if 'Estado' in df_base_estado_antiga.columns:
            df_old_active = df_base_estado_antiga[df_base_estado_antiga['Estado'] != 'Excluido'].copy()
            df_previously_excluded = df_base_estado_antiga[df_base_estado_antiga['Estado'] == 'Excluido'].copy()
        else:
            df_old_active = df_base_estado_antiga.copy()
            df_previously_excluded = pd.DataFrame()
            
        df_old_comp = df_old_active.drop(columns=['Estado'], errors='ignore')
        df_new_comp = df_novo.copy()
        
        df_old_comp = padronizar_justificativa(df_old_comp)
        
        df_old_comp_filled = df_old_comp.fillna('')
        df_new_comp_filled = df_new_comp.fillna('')
        
        df_comparado = pd.merge(df_old_comp_filled, df_new_comp_filled, how='outer', indicator=True)
        
        novas_mudancas = len(df_comparado[df_comparado['_merge'] == 'right_only']) + len(df_comparado[df_comparado['_merge'] == 'left_only'])
        
        mapa_estado = {'left_only': 'Excluido', 'right_only': 'Novo', 'both': 'Presente'}
        
        # CORREÇÃO AQUI: Padronizado para 'Estado' com E maiúsculo!
        df_comparado['Estado'] = df_comparado['_merge'].map(mapa_estado)
        df_comparado = df_comparado.drop(columns=['_merge'])
        
        if not df_previously_excluded.empty:
            df_comparado = pd.concat([df_comparado, df_previously_excluded], ignore_index=True)
            
        if novas_mudancas == 0:
            houve_alteracao_estado = False
            
    else:
        df_comparado = df_novo.copy()
        df_comparado['Estado'] = 'Novo'
        houve_alteracao_estado = True

    if houve_alteracao_estado:
        arquivo_temp_estado = destino / f"temp_estado_{nome_oficial}"
        aba_estado = "Relatorio Comparado"
        
        # Garante que a coluna Mercado não vai sujar a base nova
        if 'Mercado' in df_comparado.columns:
            df_comparado = df_comparado.drop(columns=['Mercado'])

        df_comparado.to_excel(arquivo_temp_estado, index=False, sheet_name=aba_estado)
        aplicar_formatacao_excel(arquivo_temp_estado, aba_estado)
        
        if caminho_base_nova.exists():
            shutil.copy2(str(caminho_base_nova), str(caminho_backup_base_nova))
            caminho_base_nova.unlink()
            
        shutil.move(str(arquivo_temp_estado), str(caminho_base_nova))

    # =========================================================
    # PARTE 2: BASE NORMAL (Com "Mercado" em branco)
    # =========================================================
    if callback_progresso: callback_progresso(0.8, "Processando Base Normal...")
    
    df_novo_base = df_novo.copy()
    
    # Prevenção: Garante que a coluna Estado não vaze para a base normal
    if 'Estado' in df_novo_base.columns:
        df_novo_base = df_novo_base.drop(columns=['Estado'])
        
    df_novo_base['Mercado'] = ""

    substituir_base = True
    if caminho_base.exists():
        df_base_antiga = pd.read_excel(caminho_base)
        if len(df_novo_base) <= len(df_base_antiga):
            substituir_base = False

    if substituir_base:
        arquivo_temp_base = destino / f"temp_base_{nome_oficial}"
        aba_normal = "Relatorio Revenue Sistema"
        
        df_novo_base.to_excel(arquivo_temp_base, index=False, sheet_name=aba_normal)
        aplicar_formatacao_excel(arquivo_temp_base, aba_normal)
        
        if caminho_base.exists():
            shutil.copy2(str(caminho_base), str(caminho_backup))
            caminho_base.unlink()
            
        shutil.move(str(arquivo_temp_base), str(caminho_base))

    # =========================================================
    # LIMPEZA FINAL
    # =========================================================
    if os.path.exists(arquivo_original):
        os.remove(arquivo_original)
    
    if callback_progresso: callback_progresso(0.9, "Arquivos processados com sucesso!")

meses_pt = {1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril', 5: 'Maio', 
    6: 'Junho', 7: 'Julho', 8: 'Agosto', 9: 'Setembro', 10: 'Outubro', 
    11: 'Novembro', 12: 'Dezembro'}
inicio = datetime.strptime("04/03/2026", "%d/%m/%Y")
atual = inicio
intervalos = []
intervalos.append(atual.strftime("%d/%m/%Y"))
intervalos.append(meses_pt[atual.month])
intervalos.append(atual.year)

nome_mes = intervalos[1]
ano_atual = intervalos[2]

origem = Path.home() / "Downloads"
destino = Path.home() / "Documents" / "Relatórios Revenue" / "Teste_ebus"

arquivos_encontrados = list(origem.rglob("RelatorioRevenue*.xls"))
if arquivos_encontrados:
    arquivos_encontrados.sort(key=os.path.getmtime, reverse=True)
    novo_nome = destino / f"Relatorio Revenue Completo - {nome_mes} {ano_atual}.xls"
    shutil.move(str(arquivos_encontrados[0]), str(novo_nome))
    processar_arquivos_relatorios(novo_nome, destino, nome_mes, ano_atual)
else:
    # --- ALTERAÇÃO: Adicionando log caso o arquivo não seja encontrado ---
    print(f"Arquivo não encontrado na pasta de downloads para o mês de {nome_mes}")